# GraphSAGE — Inductive Representation Learning on Large Graphs (2017)

_Papers · Graph Neural Networks_

**Learn a FUNCTION that builds a node's embedding by sampling and aggregating its neighbors, so it works on nodes never seen in training.**

---

This notebook is a practice scaffold for the **GraphSAGE — Inductive Representation Learning on Large Graphs (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- 0. Sanity-check the worked example: mean-aggregate + concat-combine + ReLU + L2-norm. ---
h_v = torch.tensor([1., 0.])
h_a, h_b = torch.tensor([0., 2.]), torch.tensor([2., 2.])
h_neigh = torch.stack([h_a, h_b]).mean(0)          # Alg.1 line 4 (mean aggregate)
cat = torch.cat([h_v, h_neigh])                    # concat(self, neigh) = [1,0,1,2]
W = torch.tensor([[1., 0., 1., 0.], [0., 1., 0., 1.]])
out = torch.relu(W @ cat)                          # Alg.1 line 5 (combine + ReLU)
out = out / out.norm()                             # Alg.1 line 7 (L2 normalize)
print("worked example:  h_neigh =", h_neigh.tolist(),
      " concat =", cat.tolist(),
      " W@cat =", (W @ cat).tolist(),
      " normalized =", [round(x, 4) for x in out.tolist()])
# worked example:  h_neigh = [1.0, 2.0]  concat = [1.0, 0.0, 1.0, 2.0]  W@cat = [2.0, 2.0]  normalized = [0.7071, 0.7071]


# --- 1. The GraphSAGE layer (built by hand): SAMPLE + mean AGGREGATE + concat-COMBINE + ReLU + L2. ---
class SAGEConv(nn.Module):
    def __init__(self, in_dim, out_dim, sample_size=8, use_neighbors=True):
        super().__init__()
        self.lin = nn.Linear(2 * in_dim, out_dim)   # W over concat(self, neigh)
        self.S   = sample_size
        self.use_neighbors = use_neighbors          # ablation switch

    def forward(self, h, adj):
        # adj[v] = list (tensor) of v's neighbor indices. h = (num_nodes, in_dim).
        agg = torch.zeros_like(h)
        for v, nbrs in enumerate(adj):
            if len(nbrs) == 0:
                agg[v] = h[v]                        # isolated node: fall back to self
                continue
            # SAMPLE: fixed-size uniform draw (with replacement so degree doesn't matter)
            idx = nbrs[torch.randint(len(nbrs), (self.S,))]
            agg[v] = h[idx].mean(0)                  # AGGREGATE (mean)
        if not self.use_neighbors:
            agg = h                                  # ablation: ignore neighbors
        out = torch.relu(self.lin(torch.cat([h, agg], dim=1)))   # COMBINE + ReLU
        return F.normalize(out, p=2, dim=1)          # L2 normalize each row


class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hid, n_classes, use_neighbors=True):
        super().__init__()
        self.l1 = SAGEConv(in_dim, hid, use_neighbors=use_neighbors)
        self.l2 = SAGEConv(hid,    hid, use_neighbors=use_neighbors)   # K = 2 layers
        self.head = nn.Linear(hid, n_classes)

    def embed(self, h, adj):
        return self.l2(self.l1(h, adj), adj)         # the inductive function: weights only, no node table
    def forward(self, h, adj):
        return self.head(self.embed(h, adj))


# --- 2. A toy 3-cluster graph. Node features are noisy; the LABEL is carried by the neighborhood. ---
def make_graph(n_per=20, seed=0):
    g = torch.Generator().manual_seed(seed)
    K = 3
    y = torch.arange(K).repeat_interleave(n_per)
    feats = torch.randn(K * n_per, 4, generator=g) * 0.3      # weak per-node features (little class signal)
    # edges: mostly within the same class -> the neighborhood, not the features, reveals the label
    adj = [[] for _ in range(K * n_per)]
    nodes_by_c = [torch.where(y == c)[0] for c in range(K)]
    for v in range(K * n_per):
        same = nodes_by_c[y[v]]
        for _ in range(6):
            u = same[torch.randint(len(same), (1,), generator=g)].item()
            if u != v: adj[v].append(u); adj[u].append(v)
    adj = [torch.tensor(sorted(set(a))) for a in adj]
    return feats, y, adj

feats, y, adj = make_graph()
N = feats.size(0)
perm = torch.randperm(N, generator=torch.Generator().manual_seed(1))
train_idx, test_idx = perm[:N // 2], perm[N // 2:]            # held-out (inductive) test nodes


# --- 3. Train on TRAINING NODES ONLY. ---
net = GraphSAGE(in_dim=4, hid=16, n_classes=3)
opt = torch.optim.Adam(net.parameters(), lr=0.03, weight_decay=5e-4)
lf  = nn.CrossEntropyLoss()
for epoch in range(120):
    net.train(); opt.zero_grad()
    logits = net(feats, adj)
    loss = lf(logits[train_idx], y[train_idx])               # supervise only training nodes
    loss.backward(); opt.step()

# --- 4. INDUCTIVE evaluation: same weights on held-out nodes (never supervised). ---
net.eval()
with torch.no_grad():
    pred = net(feats, adj).argmax(1)
    tr_acc = (pred[train_idx] == y[train_idx]).float().mean().item()
    te_acc = (pred[test_idx]  == y[test_idx]).float().mean().item()
print(f"train acc {tr_acc:.2f}   held-out (inductive) acc {te_acc:.2f}")

# --- 5. A BRAND-NEW node added AFTER training, wired to class-1 nodes. Embed with NO retraining. ---
class1 = torch.where(y == 1)[0][:4]
new_feats = torch.cat([feats, torch.randn(1, 4) * 0.3], dim=0)
new_adj   = adj + [class1]                                    # its neighbors are class-1 nodes
for u in class1: new_adj[u] = torch.cat([new_adj[u], torch.tensor([N])])
with torch.no_grad():
    new_pred = net(new_feats, new_adj).argmax(1)[N].item()
print("brand-new node predicted class:", new_pred, "(its neighbors are class 1)")
# Our small run (not the paper's numbers): train acc ~0.97, held-out (inductive) acc ~0.93,
# and the brand-new node is classified as class 1 -- the inductive property, from weights
# shared across all nodes (no per-node table, no retraining).

## Visualize the data & results

_Does sampling-and-aggregating neighbors (GraphSAGE) beat a self-only ablation when the label is carried by the neighborhood — and does it generalize to held-out, never-trained nodes (inductive)?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# Reproduces the qualitative effect on toy data: neighborhood aggregation is what lets
# GraphSAGE classify held-out (never-trained) nodes. Self-only ablation falls to ~chance.
torch.manual_seed(0)

def make_graph(n_per=20, seed=0):
    g = torch.Generator().manual_seed(seed); K = 3
    y = torch.arange(K).repeat_interleave(n_per)
    feats = torch.randn(K * n_per, 4, generator=g) * 0.3
    adj = [[] for _ in range(K * n_per)]
    by_c = [torch.where(y == c)[0] for c in range(K)]
    for v in range(K * n_per):
        same = by_c[y[v]]
        for _ in range(6):
            u = same[torch.randint(len(same), (1,), generator=g)].item()
            if u != v: adj[v].append(u); adj[u].append(v)
    return feats, y, [torch.tensor(sorted(set(a))) for a in adj]

class SAGEConv(nn.Module):
    def __init__(self, i, o, S=8, use_neighbors=True):
        super().__init__(); self.lin = nn.Linear(2 * i, o); self.S = S; self.un = use_neighbors
    def forward(self, h, adj):
        agg = torch.zeros_like(h)
        for v, nb in enumerate(adj):
            agg[v] = h[v] if len(nb) == 0 else h[nb[torch.randint(len(nb), (self.S,))]].mean(0)
        if not self.un: agg = h
        return F.normalize(torch.relu(self.lin(torch.cat([h, agg], 1))), p=2, dim=1)

class Net(nn.Module):
    def __init__(self, un=True):
        super().__init__()
        self.l1 = SAGEConv(4, 16, use_neighbors=un); self.l2 = SAGEConv(16, 16, use_neighbors=un)
        self.head = nn.Linear(16, 3)
    def forward(self, h, a): return self.head(self.l2(self.l1(h, a), a))

feats, y, adj = make_graph(); N = feats.size(0)
perm = torch.randperm(N, generator=torch.Generator().manual_seed(1))
tr, te = perm[:N // 2], perm[N // 2:]

def run(use_neighbors):
    torch.manual_seed(0)
    net = Net(use_neighbors); opt = torch.optim.Adam(net.parameters(), lr=0.03, weight_decay=5e-4)
    lf = nn.CrossEntropyLoss()
    for _ in range(120):
        net.train(); opt.zero_grad()
        lf(net(feats, adj)[tr], y[tr]).backward(); opt.step()
    net.eval()
    with torch.no_grad():
        p = net(feats, adj).argmax(1)
    return (p[te] == y[te]).float().mean().item()

print("GraphSAGE held-out acc:", round(run(True), 2))
print("Self-only  held-out acc:", round(run(False), 2))
# GraphSAGE ~0.93 ; self-only ~0.23 (chance = 0.33). Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The inductive test. You trained the two-layer GraphSAGE on a toy graph using only the
            training nodes' labels. A brand-new node arrives after training, with the same feature format and
            a few edges to existing nodes. Describe exactly what the model does to embed it, and why no
            retraining is needed.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look the new node up in its feature vector and list its edges; uniformly sample up to $S$ of its neighbors. — _The forward pass needs only the node's own features and a sample of neighbors' features &mdash; both available immediately._
- Run the two SAGE layers using the ALREADY-TRAINED shared weights $\mathbf{W}^1, \mathbf{W}^2$: aggregate, concat with self, $\mathbf{W}$, ReLU, normalize &mdash; twice. — _Because the only parameters are the shared $\mathbf{W}^k$ (no per-node row), the same function applies to a node never seen in training._
- Feed the resulting embedding to the trained classifier head to predict its class. — _The embedding lives in the same space as training embeddings, so the existing head works as-is._

**Answer:** The model samples the new node's neighbors, aggregates their features (mean),
                 combines with the new node's own features through the already-trained shared weights
                 $\mathbf{W}^k$, normalizes, and reads off the embedding &mdash; then classifies it. No
                 retraining is needed because GraphSAGE learned a function (the aggregators), not a
                 per-node lookup table. This is exactly the inductive property a transductive method
                 (paper-gcn) lacks.

</details>

**Problem 2.** Recompute the worked example with a DIFFERENT weight matrix
            $\mathbf{W} = \begin{bmatrix} 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$ (same
            $\mathbf{h}_v = [1,0]$, neighbors $[0,2]$ and $[2,2]$). What does the layer output before
            normalization, and what does this $\mathbf{W}$ do conceptually?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Aggregate: neighbor mean $= [1,2]$ (unchanged from the example). — _Same neighbors, same mean aggregator._
- Concat: $[1,0,1,2]$. Apply $\mathbf{W}$: row 1 picks column 3 $\to 1$; row 2 picks column 4 $\to 2$. Output $[1,2]$. — _This $\mathbf{W}$ zeroes the self block and copies only the neighbor-aggregate block._
- ReLU $([1,2]) = [1,2]$. — _Both positive._

**Answer:** The layer outputs $[1,\,2]$ before normalization &mdash; exactly the neighbor mean. This
                 $\mathbf{W}$ has its self-columns zeroed, so it ignores the node's own features and keeps
                 only the aggregated neighborhood. It shows concretely that the self-vs-neighbor split in
                 $\mathbf{W}\cdot\mathrm{concat}(\cdot,\cdot)$ is a learnable mix: with these weights the
                 model has learned "describe me purely by my neighbors."

</details>

**Problem 3.** Ablation. You remove the neighbor branch entirely &mdash; the layer becomes
            out = relu(W_self @ h_self) with no aggregation. On a graph where a node's class is
            determined by its neighbors (not its own features), what happens to accuracy, and what does that
            isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Strip the aggregate + concat: keep only the self vector through a linear map. — _An honest ablation changes exactly one thing &mdash; the neighborhood information &mdash; so any drop is attributable to it._
- Retrain and compare test accuracy on held-out nodes to the full model. — _If the label depends on neighbors, removing them should hurt._
- Observe accuracy fall toward chance on neighbor-determined labels. — _Without aggregation each node is judged only by its own (uninformative) features._

**Answer:** Accuracy collapses toward chance, because the label is carried by the neighborhood
                 and we deleted the aggregate. This isolates the sample-and-aggregate step as the source
                 of GraphSAGE's power: the embedding is informative precisely because it mixes in sampled
                 neighbors, not just the node's own features. The CODEVIZ panel shows this with-neighbors vs
                 self-only contrast.

</details>